# 03 - Limpieza y Transformación de Datos

**Objetivo**: Limpiar, transformar y validar los datos para análisis.

**Input**: `data/staging/{mes}_standardized.parquet`

**Output**: `data/processed/{mes}_{año}_facturas.parquet`

**Responsabilidades**:
- Limpiar nombres de productos (lowercase, trim, etc)
- Estandarizar unidades de medida
- Redondear valores decimales
- Validar datos (rangos, valores nulos críticos)
- Crear categorías de productos
- Marcar registros con problemas

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

## Configuración

In [ ]:
# Configurar el mes a procesar
MES = "enero"
ANIO = 2026

# Rutas
STAGING_PATH = Path("../data/staging")
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

INPUT_FILE = STAGING_PATH / f"{MES}_{ANIO}_standardized.csv"
OUTPUT_FILE = PROCESSED_PATH / f"{MES}_{ANIO}_facturas.csv                                          "

print(f"📂 Archivo de entrada: {INPUT_FILE}")
print(f"💾 Archivo de salida: {OUTPUT_FILE}")

📂 Archivo de entrada: ../data/staging/enero_2026_standardized.csv
💾 Archivo de salida: ../data/processed/enero_2026_facturas.csv


## 1. Cargar datos estandarizados

In [5]:
# Cargar datos
df = pd.read_csv(INPUT_FILE)

print(f"✅ Datos cargados:")
print(f"   - Filas: {len(df)}")
print(f"   - Columnas: {list(df.columns)}")

# Crear copia para limpieza
df_clean = df.copy()

df_clean.head()

✅ Datos cargados:
   - Filas: 94
   - Columnas: ['Producto', 'Cantidad', 'Unidad', 'Total', 'Valor_Unitario', 'tienda', 'fecha', 'mes', 'año']


,Producto,Cantidad,Unidad,Total,Valor_Unitario,tienda,fecha,mes,año
0,yogurt griego,1.0,und,7300.0,7300.0,d1,2026-01-11,1,2026
1,jamón pietrán,1.0,und,13650.0,13650.0,d1,2026-01-11,1,2026
2,queso parmesano,1.0,und,14850.0,14850.0,d1,2026-01-11,1,2026
3,quesillo tajado,1.0,und,9990.0,9990.0,d1,2026-01-11,1,2026
4,topping mediano,4.0,und,19800.0,4950.0,d1,2026-01-11,1,2026


## 2. Limpieza de nombres de productos

In [6]:
def limpiar_nombre_producto(nombre):
    """
    Limpia y normaliza el nombre de un producto.
    """
    if pd.isna(nombre):
        return nombre
    
    # Convertir a string
    nombre = str(nombre)
    
    # Lowercase
    nombre = nombre.lower()
    
    # Remover espacios extra
    nombre = ' '.join(nombre.split())
    
    # Remover caracteres especiales al inicio/final
    nombre = nombre.strip()
    
    return nombre

# Aplicar limpieza
print("🧹 Limpiando nombres de productos...")
df_clean['Producto'] = df_clean['Producto'].apply(limpiar_nombre_producto)

print("\n📝 Ejemplos de productos limpios:")
print(df_clean['Producto'].head(10).tolist())

🧹 Limpiando nombres de productos...

📝 Ejemplos de productos limpios:
['yogurt griego', 'jamón pietrán', 'queso parmesano', 'quesillo tajado', 'topping mediano', 'tortilla saborizada', 'desodorante', 'copitos little', 'arequipe sin', 'bolsa reciclable']


## 3. Estandarización de unidades

In [7]:
# Mapeo de unidades comunes
UNIDADES_MAP = {
    'und': 'und',
    'unidad': 'und',
    'paq': 'paq',
    'paquete': 'paq',
    'kg': 'kg',
    'kilo': 'kg',
    'kilogramo': 'kg',
    'gr': 'g',
    'g': 'g',
    'gramo': 'g',
    'gramos': 'g',
    'lt': 'l',
    'l': 'l',
    'litro': 'l',
    'ml': 'ml',
    'mililitro': 'ml'
}

def estandarizar_unidad(unidad):
    """
    Estandariza la unidad de medida.
    """
    if pd.isna(unidad):
        return unidad
    
    # Lowercase y trim
    unidad = str(unidad).lower().strip()
    
    # Buscar en mapeo
    return UNIDADES_MAP.get(unidad, unidad)

# Aplicar estandarización
print("📏 Estandarizando unidades...")
print(f"\nUnidades originales únicas: {df_clean['Unidad'].unique()}")

df_clean['Unidad'] = df_clean['Unidad'].apply(estandarizar_unidad)

print(f"\nUnidades estandarizadas únicas: {df_clean['Unidad'].unique()}")

📏 Estandarizando unidades...

Unidades originales únicas: <StringArray>
['und', 'kg', 'paquete', 'paq']
Length: 4, dtype: str

Unidades estandarizadas únicas: <StringArray>
['und', 'kg', 'paq']
Length: 3, dtype: str


## 4. Redondeo de valores numéricos

In [8]:
# Redondear valores monetarios a 2 decimales
df_clean['Valor_Unitario'] = df_clean['Valor_Unitario'].round(2)
df_clean['Total'] = df_clean['Total'].round(2)

# Redondear cantidades a 2 decimales (para productos por peso)
df_clean['Cantidad'] = df_clean['Cantidad'].round(2)

print("✅ Valores numéricos redondeados")
print("\n📊 Estadísticas de valores:")
print(df_clean[['Cantidad', 'Valor_Unitario', 'Total']].describe())

✅ Valores numéricos redondeados

📊 Estadísticas de valores:
        Cantidad  Valor_Unitario         Total
count  94.000000       94.000000     94.000000
mean    1.526170     7368.454681  11035.202128
std     0.813966     6640.269769  13228.039702
min     0.280000      200.000000    400.000000
25%     1.000000     2990.000000   4249.000000
50%     1.000000     5535.060000   7300.000000
75%     2.000000     9365.000000  10372.500000
max     4.000000    32900.290000  72563.000000


## 5. Validación de datos

In [11]:
print("🔍 Validando datos...\n")

# Crear columna de flags de problemas
df_clean['problemas'] = ''

# 1. Valores nulos en columnas críticas
mask_null_producto = df_clean['Producto'].isnull()
mask_null_cantidad = df_clean['Cantidad'].isnull()
mask_null_valor = df_clean['Valor_Unitario'].isnull()
mask_null_total = df_clean['Total'].isnull()

if mask_null_producto.any():
    df_clean.loc[mask_null_producto, 'problemas'] += 'PRODUCTO_NULL;'
    print(f"⚠️  {mask_null_producto.sum()} registros sin nombre de producto")

if mask_null_cantidad.any():
    df_clean.loc[mask_null_cantidad, 'problemas'] += 'CANTIDAD_NULL;'
    print(f"⚠️  {mask_null_cantidad.sum()} registros sin cantidad")

if mask_null_valor.any():
    df_clean.loc[mask_null_valor, 'problemas'] += 'VALOR_NULL;'
    print(f"⚠️  {mask_null_valor.sum()} registros sin valor unitario")

if mask_null_total.any():
    df_clean.loc[mask_null_total, 'problemas'] += 'TOTAL_NULL;'
    print(f"⚠️  {mask_null_total.sum()} registros sin total")

# 2. Valores negativos
mask_neg_cantidad = df_clean['Cantidad'] < 0
mask_neg_valor = df_clean['Valor_Unitario'] < 0
mask_neg_total = df_clean['Total'] < 0

if mask_neg_cantidad.any():
    df_clean.loc[mask_neg_cantidad, 'problemas'] += 'CANTIDAD_NEGATIVA;'
    print(f"⚠️  {mask_neg_cantidad.sum()} registros con cantidad negativa")

if mask_neg_valor.any():
    df_clean.loc[mask_neg_valor, 'problemas'] += 'VALOR_NEGATIVO;'
    print(f"⚠️  {mask_neg_valor.sum()} registros con valor negativo")

if mask_neg_total.any():
    df_clean.loc[mask_neg_total, 'problemas'] += 'TOTAL_NEGATIVO;'
    print(f"⚠️  {mask_neg_total.sum()} registros con total negativo")

# 3. Valores cero
mask_zero_cantidad = df_clean['Cantidad'] == 0
mask_zero_valor = df_clean['Valor_Unitario'] == 0

if mask_zero_cantidad.any():
    df_clean.loc[mask_zero_cantidad, 'problemas'] += 'CANTIDAD_CERO;'
    print(f"⚠️  {mask_zero_cantidad.sum()} registros con cantidad = 0")

if mask_zero_valor.any():
    df_clean.loc[mask_zero_valor, 'problemas'] += 'VALOR_CERO;'
    print(f"⚠️  {mask_zero_valor.sum()} registros con valor = 0")

# 4. Inconsistencia en cálculo de Total
df_clean['total_calculado'] = df_clean['Cantidad'] * df_clean['Valor_Unitario']
df_clean['diferencia_total'] = abs(df_clean['Total'] - df_clean['total_calculado'])
mask_inconsistente = df_clean['diferencia_total'] >  300 # Tolerancia de 300 pesos

if mask_inconsistente.any():
    df_clean.loc[mask_inconsistente, 'problemas'] += 'TOTAL_INCONSISTENTE;'
    print(f"⚠️  {mask_inconsistente.sum()} registros con total inconsistente")

# Resumen
total_problemas = (df_clean['problemas'] != '').sum()
print(f"\n📊 Total de registros con problemas: {total_problemas} ({total_problemas/len(df_clean)*100:.2f}%)")
print(f"✅ Registros limpios: {len(df_clean) - total_problemas} ({(len(df_clean) - total_problemas)/len(df_clean)*100:.2f}%)")

🔍 Validando datos...


📊 Total de registros con problemas: 0 (0.00%)
✅ Registros limpios: 94 (100.00%)


## 6. Revisar registros con problemas

In [12]:
# Mostrar registros con problemas
df_problemas = df_clean[df_clean['problemas'] != '']

if len(df_problemas) > 0:
    print(f"⚠️  Registros con problemas ({len(df_problemas)}):")
    print(df_problemas[['Producto', 'Cantidad', 'Valor_Unitario', 'Total', 'problemas']].head(10))
else:
    print("✅ No se encontraron registros con problemas")

✅ No se encontraron registros con problemas


## 7. Categorización de productos (opcional)

Clasificar productos en categorías basadas en palabras clave

In [ ]:
def categorizar_producto(nombre):
    """
    Asigna una categoría al producto basada en palabras clave.
    """
    if pd.isna(nombre):
        return 'Sin categoría'
    
    nombre = str(nombre).lower()
    
    # Lácteos
    if any(palabra in nombre for palabra in ['leche', 'queso', 'yogurt', 'quesillo', 'kumis']):
        return 'Lácteos'
    
    # Carnes y embutidos
    if any(palabra in nombre for palabra in ['jamón', 'salchicha', 'pollo', 'carne', 'chorizo']):
        return 'Carnes y embutidos'
    
    # Frutas y verduras
    if any(palabra in nombre for palabra in ['papaya', 'manzana', 'plátano', 'naranja', 'tomate', 'lechuga', 'cebolla', 'champiñón']):
        return 'Frutas y verduras'
    
    # Granos y cereales
    if any(palabra in nombre for palabra in ['arroz', 'pasta', 'avena', 'cereal', 'pan']):
        return 'Granos y cereales'
    
    # Bebidas
    if any(palabra in nombre for palabra in ['jugo', 'gaseosa', 'agua', 'té', 'café']):
        return 'Bebidas'
    
    # Despensa
    if any(palabra in nombre for palabra in ['aceite', 'sal', 'azúcar', 'harina', 'topping']):
        return 'Despensa'
    
    # Aseo
    if any(palabra in nombre for palabra in ['jabón', 'shampoo', 'detergente', 'papel']):
        return 'Aseo'
    
    return 'Otros'

# Aplicar categorización
print("🏷️  Categorizando productos...")
df_clean['Categoria'] = df_clean['Producto'].apply(categorizar_producto)

print("\n📊 Distribución por categoría:")
print(df_clean['Categoria'].value_counts())

## 8. Seleccionar columnas finales

In [ ]:
# Columnas finales para el dataset procesado
columnas_finales = [
    'Producto',
    'Categoria',
    'Cantidad',
    'Unidad',
    'Valor_Unitario',
    'Total',
    'tienda',
    'fecha',
    'mes',
    'año',
    'problemas'
]

df_final = df_clean[columnas_finales].copy()

print("📋 Columnas finales del dataset:")
print(columnas_finales)

df_final.head()

## 9. Estadísticas finales

In [ ]:
print("📊 ESTADÍSTICAS DEL DATASET FINAL\n")
print("=" * 50)

print(f"\n📦 Total de registros: {len(df_final)}")
print(f"🏪 Tiendas: {df_final['tienda'].nunique()}")
print(df_final['tienda'].value_counts())

print(f"\n📅 Rango de fechas: {df_final['fecha'].min()} a {df_final['fecha'].max()}")
print(f"📅 Total de días con compras: {df_final['fecha'].nunique()}")

print(f"\n🛒 Productos únicos: {df_final['Producto'].nunique()}")

print("\n🏷️  Distribución por categoría:")
print(df_final['Categoria'].value_counts())

print(f"\n💰 Total gastado: ${df_final['Total'].sum():,.2f}")
print(f"💵 Promedio por compra: ${df_final.groupby('fecha')['Total'].sum().mean():,.2f}")
print(f"📊 Ticket promedio por producto: ${df_final['Total'].mean():,.2f}")

print("\n🔍 Top 10 productos más comprados:")
print(df_final['Producto'].value_counts().head(10))

print("\n💎 Top 10 productos más caros (valor unitario):")
print(df_final.nlargest(10, 'Valor_Unitario')[['Producto', 'Valor_Unitario', 'tienda']])

print("\n⚠️  Registros con problemas:")
print(f"Total: {(df_final['problemas'] != '').sum()} ({(df_final['problemas'] != '').sum()/len(df_final)*100:.2f}%)")

## 10. Guardar dataset procesado

In [ ]:
# Guardar como parquet
df_final.to_parquet(OUTPUT_FILE, index=False)

print(f"\n💾 Archivo guardado en: {OUTPUT_FILE}")
print(f"   - Tamaño: {OUTPUT_FILE.stat().st_size / 1024:.2f} KB")
print(f"   - Filas: {len(df_final)}")
print(f"   - Columnas: {len(df_final.columns)}")

# También guardar versión CSV para fácil visualización
csv_file = OUTPUT_FILE.with_suffix('.csv')
df_final.to_csv(csv_file, index=False)
print(f"\n📄 También guardado en CSV: {csv_file}")

print(f"\n✅ LIMPIEZA Y TRANSFORMACIÓN COMPLETADA")

## Resumen

Este notebook:
1. ✅ Cargó datos estandarizados de staging
2. ✅ Limpió nombres de productos (lowercase, trim)
3. ✅ Estandarizó unidades de medida
4. ✅ Redondeó valores numéricos
5. ✅ Validó datos y marcó problemas
6. ✅ Categorizó productos por tipo
7. ✅ Generó estadísticas del dataset
8. ✅ Guardó dataset procesado listo para análisis

**Siguiente paso**: 
- Ejecutar `04_eda.ipynb` para análisis exploratorio
- Conectar a Power BI usando el archivo procesado